# 작물 질병 이미지 사전 전처리 — Colab 버전

**Drive에서 원본 zip 직접 read + Colab 로컬에 결과 저장 + 완성 zip만 Drive 업로드**

## 실행 전 준비

- [ ] Drive에 `preprocessing_code/` 폴더 업로드 (헬퍼 .py 파일들)
  - 위치 예: `G:\내 드라이브\판다넘_공유\preprocessing_code\`
  - 포함 파일: `config.py` (이건 이 노트북과 함께 제공된 Colab용), `label_utils.py`, `crop_utils.py`, `split_utils.py`, `sampling_utils.py`
- [ ] Drive에 `datasets/` 폴더 (이미 있음, 원본 zip)
- [ ] Drive에 `deeplearning_labeling.zip` (이미 있음)
- [ ] 결과 zip 모을 폴더: `판다넘_공유/preprocessed/` (미리 만들기)

## 작업 흐름

1. Drive 마운트
2. 헬퍼 모듈 로드 (Drive에 올린 .py들)
3. 작물 1개 선택 → 처리 → zip → Drive 업로드 → 다음 작물
4. Colab 디스크 정리 (다음 작물 위해)

---
## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 헬퍼 모듈 경로 추가

Drive에 올린 `preprocessing_code/` 폴더 위치를 `sys.path` 에 추가.
**자기 Drive의 실제 경로로 수정하세요.**

In [ ]:
import sys

# Drive에 업로드한 preprocessing_code 폴더 경로 (자기 환경에 맞게 수정)
CODE_DIR = '/content/drive/MyDrive/Colab_PandaNum'

sys.path.insert(0, CODE_DIR)

# 확인
from pathlib import Path
assert Path(CODE_DIR).exists(), f'❌ 경로 없음: {CODE_DIR}'
for fname in ['config.py', 'label_utils.py', 'crop_utils.py', 'split_utils.py', 'sampling_utils.py']:
    fpath = Path(CODE_DIR) / fname
    print(f'  {fname}: {"✓" if fpath.exists() else "✗ 없음!"}')
print('\n경로 추가 완료')

  config.py: ✓
  label_utils.py: ✓
  crop_utils.py: ✓
  split_utils.py: ✓
  sampling_utils.py: ✓

경로 추가 완료


## 3. 패키지 설치 (한 번만)

Colab엔 보통 다 있지만 안전하게 확인.

In [ ]:
!pip install opencv-python-headless scikit-learn -q
print('패키지 OK')

패키지 OK


## 4. 모듈 import

In [ ]:
import csv
import time
import zipfile
import shutil
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

from config import (
    RAW_DATA_ROOT, LABEL_ZIP_PATH, OUTPUT_ROOT,
    ENV_EN, SUBFOLDER_TRAIN, SUBFOLDER_VAL,
    MAX_PER_CLASS, TARGET_SIZE, JPEG_QUALITY, PADDING_RATIO,
    SPLIT_RATIO, RANDOM_SEED, RISK_NAMES,
    TRAIN_GROUPS, HELDOUT_GROUPS,
    group_id, group_type_of,
)
from label_utils    import LabelInfo, load_group_labels, summarize_labels
from split_utils    import assign_splits, split_distribution
from sampling_utils import apply_cap, cap_summary
from crop_utils     import preprocess_image
from tqdm.notebook  import tqdm

print('모듈 로드 완료')
print(f'RAW_DATA_ROOT : {RAW_DATA_ROOT}  ({"존재" if RAW_DATA_ROOT.exists() else "없음"})')
print(f'LABEL_ZIP_PATH: {LABEL_ZIP_PATH}  ({"존재" if LABEL_ZIP_PATH.exists() else "없음"})')
print(f'OUTPUT_ROOT   : {OUTPUT_ROOT}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

모듈 로드 완료
RAW_DATA_ROOT : /content/drive/MyDrive/datasets  (존재)
LABEL_ZIP_PATH: /content/drive/MyDrive/deeplearning_labeling.zip  (존재)
OUTPUT_ROOT   : /content/preprocessed


## 5. 결과 업로드 위치

처리한 zip을 올릴 Drive 폴더 (공유 폴더). **자기 환경에 맞게 수정.**

In [ ]:
UPLOAD_DIR = Path('/content/drive/MyDrive/datasets/PandaNum/preprocessed')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f'업로드 위치: {UPLOAD_DIR}')

업로드 위치: /content/drive/MyDrive/datasets/PandaNum/preprocessed


---
## 6. 함수 정의 (그대로 실행)

In [ ]:
def find_source_zips(env: str, crop_folder: str) -> Dict[str, List[Path]]:
    """원본 zip 탐색 (Colab Drive 마운트 안전 버전).

    glob 패턴 대신 명시적 경로 탐색 — 한글 폴더명에서도 안정적.
    """
    result = {'train': [], 'val': []}
    if not RAW_DATA_ROOT.exists():
        print(f'  ⚠ RAW_DATA_ROOT 없음: {RAW_DATA_ROOT}')
        return result

    env_code = '071' if env == '시설' else '073'

    # ── 패턴 1: AI Hub 원본 구조 (명시적 탐색) ──
    for top_dir in RAW_DATA_ROOT.iterdir():
        if not top_dir.is_dir():
            continue
        for ds_dir in top_dir.iterdir():
            if not ds_dir.is_dir():
                continue
            if not ds_dir.name.startswith(f'{env_code}.'):
                continue
            data_dir = ds_dir / '01.데이터'
            if not data_dir.exists():
                continue
            for sub_aihub, key in [('1.Training', 'train'), ('2.Validation', 'val')]:
                crop_path = data_dir / sub_aihub / '원천데이터' / crop_folder
                if crop_path.exists():
                    result[key].extend(crop_path.glob('*.zip'))

    # ── 패턴 2: 표준 구조 ──
    base_std = RAW_DATA_ROOT / ENV_EN[env] / crop_folder
    for sub, key in [(SUBFOLDER_TRAIN, 'train'), (SUBFOLDER_VAL, 'val')]:
        sub_path = base_std / sub
        if sub_path.exists():
            result[key].extend(sub_path.glob('*.zip'))

    # 중복 제거 + 정렬
    result['train'] = sorted(set(result['train']))
    result['val']   = sorted(set(result['val']))
    return result


In [ ]:
def process_group(env: str, crop_folder: str) -> Optional[Dict]:
    """한 그룹 전처리."""
    gtype = group_type_of(env, crop_folder)
    gid   = group_id(env, crop_folder)

    print('\n' + '='*70)
    print(f'▶ [{gtype}] {env} {crop_folder}  →  {gid}')
    print('='*70)
    t0 = time.time()

    # 1. 라벨 로드
    print('\n[1/5] 라벨 로드 중...')
    labels = load_group_labels(LABEL_ZIP_PATH, env, crop_folder)
    if not labels:
        print('  ❌ 라벨 없음.')
        return None
    summary = summarize_labels(labels)
    print(f"  → 총 {summary['total']:,}장 (원본 {summary['is_aug_false']:,} + 증강 {summary['is_aug_true']:,})")

    # 2. 분할
    print('\n[2/5] D-1 분할...')
    splits = assign_splits(labels, gtype)
    dist = split_distribution(splits)
    for split, info in dist.items():
        risks = {RISK_NAMES[k]: v for k, v in sorted(info['by_risk'].items())}
        print(f"  {split:18s}: {info['total']:>6,}장  {risks}")

    # 3. 캡
    print(f'\n[3/5] 캡 {MAX_PER_CLASS} 적용...')
    capped = apply_cap(splits) if gtype == 'train_group' else splits
    after = split_distribution(capped)
    for split, info in after.items():
        risks = {RISK_NAMES[k]: v for k, v in sorted(info['by_risk'].items())}
        print(f"  {split:18s}: {info['total']:>6,}장  {risks}")

    # 4. 원본 zip 탐색
    print('\n[4/5] 원본 zip 탐색...')
    src_zips = find_source_zips(env, crop_folder)
    n_zips = sum(len(v) for v in src_zips.values())
    if n_zips == 0:
        print(f'  ❌ {RAW_DATA_ROOT} 에 zip 없음.')
        return None
    for k, zips in src_zips.items():
        print(f'  {k}: {len(zips)}개')
        for z in zips:
            size_gb = z.stat().st_size / (1024**3)
            print(f'    - {z.name} ({size_gb:.2f} GB)')

    # 5. 처리
    print('\n[5/5] 이미지 전처리 + 저장...')
    out_dir = OUTPUT_ROOT / gid
    img_dir = out_dir / 'images'
    img_dir.mkdir(parents=True, exist_ok=True)

    lookup = {l.image_filename: (l, s) for l, s in capped}
    manifest_rows = []
    counter = skipped_no_label = skipped_decode = 0

    for split_aihub, zip_list in src_zips.items():
        for zip_path in zip_list:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                entries = [e for e in zf.infolist()
                           if Path(e.filename).name.lower().endswith(('.jpg', '.jpeg', '.png'))]
                for entry in tqdm(entries, desc=zip_path.name, leave=False):
                    name = Path(entry.filename).name
                    if name not in lookup:
                        skipped_no_label += 1
                        continue
                    info, split = lookup[name]
                    with zf.open(entry) as f:
                        img_bytes = f.read()
                    processed = preprocess_image(img_bytes, info.bbox)
                    if processed is None:
                        skipped_decode += 1
                        continue
                    counter += 1
                    out_name = f'img_{counter:07d}.jpg'
                    with open(img_dir / out_name, 'wb') as f:
                        f.write(processed)
                    manifest_rows.append({
                        'file':        out_name,
                        'env':         info.env,
                        'crop_folder': info.crop_folder,
                        'crop_code':   info.crop_code,
                        'disease':     info.disease,
                        'risk':        info.risk,
                        'grow':        info.grow,
                        'original_id': info.original_id,
                        'is_aug':      info.is_aug,
                        'split':       split,
                        'group_type':  gtype,
                        'group_id':    gid,
                    })

    if manifest_rows:
        keys = list(manifest_rows[0].keys())
        with open(out_dir / 'manifest.csv', 'w', encoding='utf-8-sig', newline='') as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(manifest_rows)

    elapsed = time.time() - t0
    print(f'\n  ✓ {counter:,}장 저장 (스킵: 라벨없음 {skipped_no_label:,} / 디코드 {skipped_decode:,})')
    print(f'  소요: {elapsed/60:.1f}분')

    return {
        'group_id': gid, 'env': env, 'crop': crop_folder, 'group_type': gtype,
        'n_images': counter, 'elapsed_min': elapsed/60,
    }

In [ ]:
def zip_group_result(env: str, crop_folder: str) -> Optional[Path]:
    """전처리 결과를 zip으로 묶기 (Colab 로컬)."""
    gid = group_id(env, crop_folder)
    src = OUTPUT_ROOT / gid
    if not src.exists():
        return None
    zip_out = OUTPUT_ROOT / f'{gid}.zip'
    print(f'\nzip 압축: {zip_out.name} ...', end='', flush=True)
    t0 = time.time()
    with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_STORED) as zf:
        for p in src.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(src.parent))
    size_mb = zip_out.stat().st_size / (1024**2)
    print(f' 완료 ({time.time()-t0:.1f}s, {size_mb:.1f} MB)')
    return zip_out

In [ ]:
def upload_to_drive(env: str, crop_folder: str) -> Optional[Path]:
    """만든 zip을 Drive 공유 폴더로 업로드."""
    gid = group_id(env, crop_folder)
    src_zip = OUTPUT_ROOT / f'{gid}.zip'
    if not src_zip.exists():
        print(f'❌ zip 없음: {src_zip}')
        return None
    dst = UPLOAD_DIR / f'{gid}.zip'
    print(f'\nDrive 업로드: {dst} ...', end='', flush=True)
    t0 = time.time()
    shutil.copy(src_zip, dst)
    print(f' 완료 ({time.time()-t0:.1f}s)')
    return dst

In [ ]:
def cleanup_local(env: str, crop_folder: str):
    """다음 작물 처리 위해 Colab 로컬 디스크 정리."""
    gid = group_id(env, crop_folder)
    src = OUTPUT_ROOT / gid
    zip_out = OUTPUT_ROOT / f'{gid}.zip'
    if src.exists():
        shutil.rmtree(src)
        print(f'삭제: {src}')
    if zip_out.exists():
        zip_out.unlink()
        print(f'삭제: {zip_out}')
    # 디스크 여유 확인
    import os
    stat = os.statvfs('/content')
    free_gb = stat.f_bavail * stat.f_frsize / (1024**3)
    print(f'Colab 디스크 여유: {free_gb:.1f} GB')

---
## 7. 작물 1개씩 처리 (Colab 디스크 용량 때문)

**한 셀당 한 작물.** 끝나면 다음 셀 실행. 한 작물 끝나면 자동으로:
1. 전처리
2. zip 묶기
3. Drive 업로드
4. 로컬 정리

한 작물 평균 1시간 이내. 무료 Colab 세션 12시간 한도 안에 충분.

### 7-1. 노지 07.잎마름병(토마토) — 가장 작음, 먼저

In [ ]:
import os

ROOT = '/content/drive/MyDrive/datasets'

print('1. Root 내용:')
for item in sorted(os.listdir(ROOT)):
    print(f'  {item}')

print('\n2. 노지_토마토 내부:')
path = f'{ROOT}/노지_토마토'
if os.path.exists(path):
    for item in sorted(os.listdir(path)):
        print(f'  {item}')

print('\n3. 잎마름병토마토 zip 직접 확인:')
deep = f'{ROOT}/노지_토마토/073.노지_작물_질병_진단/01.데이터/1.Training/원천데이터/07.잎마름병(토마토)'
if os.path.exists(deep):
    for item in sorted(os.listdir(deep)):
        size_mb = os.path.getsize(f'{deep}/{item}') / 1024 / 1024
        print(f'  {item} ({size_mb:.1f} MB)')
else:
    print(f'  ❌ 경로 없음: {deep}')


1. Root 내용:
  PandaNum
  confusion_matrix_tomato.png
  efficientnet_b0_tomato_pilot.pth
  learning_curve_tomato.png
  preprocessed
  노지_배추
  노지_토마토
  노지_호박
  시설_참외

2. 노지_토마토 내부:
  073.노지_작물_질병_진단

3. 잎마름병토마토 zip 직접 확인:
  0.정상.zip (18497.8 MB)
  1.질병.zip (157.3 MB)
  9.증강.zip (3469.5 MB)


In [ ]:
env, crop = '노지', '07.잎마름병(토마토)'

process_group(env, crop)
zip_group_result(env, crop)
upload_to_drive(env, crop)
cleanup_local(env, crop)


▶ [heldout_group] 노지 07.잎마름병(토마토)  →  outdoor_07_잎마름병토마토

[1/5] 라벨 로드 중...
  → 총 16,998장 (원본 13,293 + 증강 3,705)

[2/5] D-1 분할...
  heldout           : 15,070장  {'정상': 11614, '초기': 1104, '중기': 1824, '말기': 528}
  heldout_external  :  1,928장  {'정상': 1432, '초기': 144, '중기': 240, '말기': 112}

[3/5] 캡 5000 적용...
  heldout           : 15,070장  {'정상': 11614, '초기': 1104, '중기': 1824, '말기': 528}
  heldout_external  :  1,928장  {'정상': 1432, '초기': 144, '중기': 240, '말기': 112}

[4/5] 원본 zip 탐색...
  train: 3개
    - 0.정상.zip (18.06 GB)
    - 1.질병.zip (0.15 GB)
    - 9.증강.zip (3.39 GB)
  val: 3개
    - 0.정상.zip (2.23 GB)
    - 1.질병.zip (0.02 GB)
    - 9.증강.zip (0.40 GB)

[5/5] 이미지 전처리 + 저장...


0.정상.zip:   0%|          | 0/11614 [00:00<?, ?it/s]

1.질병.zip:   0%|          | 0/216 [00:00<?, ?it/s]

9.증강.zip:   0%|          | 0/3240 [00:00<?, ?it/s]

0.정상.zip:   0%|          | 0/1432 [00:00<?, ?it/s]

1.질병.zip:   0%|          | 0/31 [00:00<?, ?it/s]

9.증강.zip:   0%|          | 0/465 [00:00<?, ?it/s]


  ✓ 16,998장 저장 (스킵: 라벨없음 0 / 디코드 0)
  소요: 33.4분

zip 압축: outdoor_07_잎마름병토마토.zip ... 완료 (26.2s, 1035.6 MB)

Drive 업로드: /content/drive/MyDrive/datasets/PandaNum/preprocessed/outdoor_07_잎마름병토마토.zip ... 완료 (9.4s)
삭제: /content/preprocessed/outdoor_07_잎마름병토마토
삭제: /content/preprocessed/outdoor_07_잎마름병토마토.zip
Colab 디스크 여유: 80.1 GB


### 7-2. 노지 10.호박

In [ ]:
env, crop = '노지', '10.호박'

process_group(env, crop)
zip_group_result(env, crop)
upload_to_drive(env, crop)
cleanup_local(env, crop)


▶ [heldout_group] 노지 10.호박  →  outdoor_10_호박

[1/5] 라벨 로드 중...
  → 총 19,719장 (원본 12,354 + 증강 7,365)

[2/5] D-1 분할...
  heldout           : 17,520장  {'정상': 10576, '초기': 1344, '중기': 1840, '말기': 3760}
  heldout_external  :  2,199장  {'정상': 1303, '초기': 192, '중기': 160, '말기': 544}

[3/5] 캡 5000 적용...
  heldout           : 17,520장  {'정상': 10576, '초기': 1344, '중기': 1840, '말기': 3760}
  heldout_external  :  2,199장  {'정상': 1303, '초기': 192, '중기': 160, '말기': 544}

[4/5] 원본 zip 탐색...
  train: 3개
    - 0.정상.zip (17.41 GB)
    - 1.질병.zip (0.41 GB)
    - 9.증강.zip (9.13 GB)
  val: 3개
    - 0.정상.zip (2.15 GB)
    - 1.질병.zip (0.06 GB)
    - 9.증강.zip (1.36 GB)

[5/5] 이미지 전처리 + 저장...


0.정상.zip:   0%|          | 0/10560 [00:00<?, ?it/s]

1.질병.zip:   0%|          | 0/435 [00:00<?, ?it/s]

9.증강.zip:   0%|          | 0/6525 [00:00<?, ?it/s]

0.정상.zip:   0%|          | 0/1303 [00:00<?, ?it/s]

1.질병.zip:   0%|          | 0/56 [00:00<?, ?it/s]

9.증강.zip:   0%|          | 0/840 [00:00<?, ?it/s]


  ✓ 19,719장 저장 (스킵: 라벨없음 0 / 디코드 0)
  소요: 39.5분

zip 압축: outdoor_10_호박.zip ... 완료 (29.5s, 1573.7 MB)

Drive 업로드: /content/drive/MyDrive/datasets/PandaNum/preprocessed/outdoor_10_호박.zip ... 완료 (11.3s)
삭제: /content/preprocessed/outdoor_10_호박
삭제: /content/preprocessed/outdoor_10_호박.zip
Colab 디스크 여유: 72.6 GB


### 7-3. 노지 03.배추 — 가장 큼, 마지막

In [ ]:
env, crop = '노지', '03.배추'

process_group(env, crop)
zip_group_result(env, crop)
upload_to_drive(env, crop)
cleanup_local(env, crop)


▶ [train_group] 노지 03.배추  →  outdoor_03_배추

[1/5] 라벨 로드 중...
  → 총 33,013장 (원본 11,908 + 증강 21,105)

[2/5] D-1 분할...
  train             : 20,634장  {'정상': 6522, '초기': 6560, '중기': 4912, '말기': 2640}
  test              :  4,422장  {'정상': 1398, '초기': 1408, '중기': 1056, '말기': 560}
  val               :  4,422장  {'정상': 1398, '초기': 1392, '중기': 1056, '말기': 576}
  external_val      :  3,535장  {'정상': 1183, '초기': 1104, '중기': 816, '말기': 432}

[3/5] 캡 5000 적용...
  train             : 17,552장  {'정상': 5000, '초기': 5000, '중기': 4912, '말기': 2640}
  test              :  4,422장  {'정상': 1398, '초기': 1408, '중기': 1056, '말기': 560}
  val               :  4,422장  {'정상': 1398, '초기': 1392, '중기': 1056, '말기': 576}
  external_val      :  3,535장  {'정상': 1183, '초기': 1104, '중기': 816, '말기': 432}

[4/5] 원본 zip 탐색...
  train: 4개
    - 0.정상.zip (17.58 GB)
    - 1.질병.zip (2.39 GB)
    - 9.증강_(1).zip (28.72 GB)
    - 9.증강_(2).zip (25.25 GB)
  val: 3개
    - 0.정상.zip (2.19 GB)
    - 1.질병.zip (0.27 GB)
    

0.정상.zip:   0%|          | 0/9318 [00:00<?, ?it/s]

1.질병.zip:   0%|          | 0/1260 [00:00<?, ?it/s]

9.증강_(1).zip:   0%|          | 0/10945 [00:00<?, ?it/s]

9.증강_(2).zip:   0%|          | 0/7955 [00:00<?, ?it/s]

0.정상.zip:   0%|          | 0/1183 [00:00<?, ?it/s]

1.질병.zip:   0%|          | 0/147 [00:00<?, ?it/s]

9.증강.zip:   0%|          | 0/2205 [00:00<?, ?it/s]


  ✓ 29,931장 저장 (스킵: 라벨없음 3,082 / 디코드 0)
  소요: 88.8분

zip 압축: outdoor_03_배추.zip ... 완료 (44.8s, 2261.3 MB)

Drive 업로드: /content/drive/MyDrive/datasets/PandaNum/preprocessed/outdoor_03_배추.zip ... 완료 (11.2s)
삭제: /content/preprocessed/outdoor_03_배추
삭제: /content/preprocessed/outdoor_03_배추.zip
Colab 디스크 여유: 76.4 GB


---
## 8. 끝!

Drive 공유 폴더에 zip 4개 (참외 이미 있고 + 잎마름병토마토/호박/배추) 모이면 끝.